# 🧠 SecureTransac AI AV Model Training
This notebook generates synthetic blockchain behavioral data and trains a Neural Network (MLP) to predict AV Scores. The model weights are then exported for use in the Node.js backend.

In [1]:
import json
import random
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## 1. Data Generation (Synthetic)
We simulate user behaviors based on:
- Transaction Volume (Normalized)
- Frequency
- Account Age
- Report History
- Social Graph Score

In [2]:
def generate_sample():
    # Healthy User simulation
    if random.random() > 0.3:
        tx_vol = random.uniform(0.5, 1.0)
        tx_freq = random.uniform(0.3, 0.8)
        age = random.uniform(0.5, 1.0)
        reports = 0
        social = random.uniform(0.6, 1.0)
        
        # Weighted scoring logic for ground truth
        score = (tx_vol * 0.2) + (tx_freq * 0.1) + (age * 0.3) + (social * 0.4)
        score += random.uniform(-0.05, 0.05) # Noise
    # Malicious User / Sybil simulation
    else:
        tx_vol = random.uniform(0.0, 0.3)
        tx_freq = random.uniform(0.8, 1.0) # Spamming behavior
        age = random.uniform(0.0, 0.2) # New account
        reports = random.choice([0, 1, 0, 0])
        social = random.uniform(0.0, 0.4)
        
        score = (tx_vol * 0.2) + (tx_freq * -0.2) + (age * 0.3) + (social * 0.4) - (reports * 0.5)
        score += random.uniform(-0.05, 0.05)
    
    return [tx_vol, tx_freq, age, reports, social], max(0.0, min(1.0, score))

X = []
y = []
for _ in range(5000):
    feat, target = generate_sample()
    X.append(feat)
    y.append(target)
    
print(f"Generated {len(X)} samples")

Generated 5000 samples


## 2. Model Training
We use a Multi-Layer Perceptron (MLP) Regressor.

In [4]:
model = MLPRegressor(hidden_layer_sizes=(8, 4), activation='relu', solver='adam', max_iter=1000)
model.fit(X, y)

print(f"Model R2 Score: {model.score(X, y):.4f}")

Model R2 Score: 0.9692


## 3. Export Weights for Backend
We extract the weights and biases to load specifically in our Node.js `NeuralNetwork` class.

In [5]:
weights = {
    "layers": []
}

for i in range(len(model.coefs_)):
    layer_data = {
        "weights": model.coefs_[i].tolist(),
        "biases": model.intercepts_[i].tolist(),
        "activation": "relu" if i < len(model.coefs_) - 1 else "identity"
    }
    weights["layers"].append(layer_data)

with open('model_weights.json', 'w') as f:
    json.dump(weights, f)
    
print("Saved model_weights.json")

Saved model_weights.json
